In [2]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import gc
import fiona
from shapely.geometry import box
import random

In [ ]:
# break conus gpkg file by vpu
RUN_THIS_BLOCK = False

if RUN_THIS_BLOCK:
    conus_file = Path("/home/yuqiong.liu/work/data/gpkg_v2.2/conus_nextgen.gpkg")
    gdf = gpd.read_file(conus_file, layer="divides")

    # Group by 'vpuid' and save each group to a new GPKG file
    for vpuid, group in gdf.groupby("vpuid"):
        out_file = f"{conus_file.parent}/vpu_divides/vpu_{vpuid}.gpkg"
        group.to_file(out_file, driver="GPKG", layer="divides")
        print(f"Saved: {out_file}")

    del gdf
    gc.collect()

In [ ]:
# for each polygon in shp2, find the corresponding polygon(s) in shp1 and the overlapping area percentage
def create_cwt(
    shp1: gpd.GeoDataFrame,
    shp2: gpd.GeoDataFrame,
    overlap_threshold: float = 99.0,
    overlap_threshold_min=20.0,
):
    # Convert to a projected CRS for accurate area calculations
    projected_crs = "EPSG:3857"
    shp1 = shp1.to_crs(projected_crs)
    shp2 = shp2.to_crs(projected_crs)

    # Find overlapping areas with shp1
    overlap = gpd.overlay(shp2, shp1, how="intersection")

    # Compute area of each original polygon in shp2 (and convert to km^2)
    shp2["original_area"] = shp2.geometry.area / 1_000_000

    # Compute intersection area (and convert to km^2)
    overlap["overlap_area"] = overlap.geometry.area / 1_000_000
    overlap = overlap.drop(columns="areasqkm")

    # merge overlap with shp2 (to get original_area)
    overlap = overlap.merge(shp2.drop(columns="geometry"), on="divide_id", how="left")

    # Calculate percentage of each shp2 polygon that is covered by intersecting polygons in shp1
    overlap["overlap_percentage"] = (
        overlap["overlap_area"] / overlap["original_area"]
    ) * 100

    # sort by divide_id and descending overlap_percentage
    overlap = overlap.sort_values(
        by=["divide_id", "overlap_percentage"], ascending=[True, False]
    )

    # compute cumulative overlap_percentage for each divide
    overlap["cum_percentage"] = overlap.groupby("divide_id")[
        "overlap_percentage"
    ].cumsum()

    # filter out shp1 polygons when cumulative overlap_percentage reaches the threshold
    # (but always keep the first row)
    overlap["rank"] = overlap.groupby("divide_id").cumcount()
    overlap = overlap[
        (overlap["rank"] == 0) | (overlap["cum_percentage"] <= overlap_threshold)
    ]

    # make sure (maximum) cumulative overlap percentage is greater than the minimum threshold
    # Find ids where max(cum_area) > threshold
    polys_matched = overlap.groupby("divide_id")["cum_percentage"].max()
    polys_matched = polys_matched[polys_matched >= overlap_threshold_min].index
    overlap = overlap[overlap["divide_id"].isin(polys_matched)]

    # identify shp2 polygons not paired with a shp1 polygon
    polys = shp2["divide_id"].unique()
    polys_unmatched = [x for x in polys if x not in polys_matched]

    # drop columns that are no longer needed
    overlap = overlap.drop(columns=["overlap_area", "original_area", "cum_percentage"])

    # Find the nearest shp1 polygon for each unmatched shp2 polygon
    nearest_matches = gpd.sjoin_nearest(
        shp2[shp2["divide_id"].isin(polys_unmatched)],
        shp1,
        how="left",
        distance_col="nearest_distance",
    )
    nearest_matches.rename(columns={"nearest_distance": "nearest_dist_m"}, inplace=True)
    nearest_matches = nearest_matches[["divide_id", "id", "nearest_dist_m"]]

    # create the final huc12/divide_id crosswalk table
    cwt = pd.concat(
        [overlap.drop(columns="geometry"), nearest_matches], axis=0, ignore_index=True
    )

    return cwt

In [4]:
def get_attr_info(attr: str) -> dict:
    attr_info = dict()
    match attr.lower():
        case "hlr":
            attr_info["file"] = "/home/yuqiong.liu/work/data/HLR/hlrshape/hlrus.shp"
            attr_info["id_column"] = "VALUE"
            attr_info["layer"] = "hlrus"
        case "hydroatlas":
            attr_info["file"] = (
                "/home/yuqiong.liu/work/data/HydroATLAS/BasinATLAS_v10_shp/BasinATLAS_v10_lev12.shp"
            )
            attr_info["id_column"] = "PFAF_ID"
            attr_info["layer"] = "BasinATLAS_v10_lev12"
        case _:
            raise Exception(f"Unsupported attribute dataset: {attr}")

    return attr_info

In [26]:
# process crosswalk table for NextGen catchments by VPU
def process_cwt_by_vpu(
    attr: str, cwt_dir: Path, threshold: float = 99.0, threshold_min=30.0
) -> pd.DataFrame:
    # get the attribute dataset info
    attr_dict = get_attr_info(attr)

    vpu_files = list(
        Path("/home/yuqiong.liu/work/data/gpkg_v2.2/vpu_divides").glob("*.gpkg")
    )
    df_cwt = pd.DataFrame()
    for vpu_file in vpu_files:
        # get the vpu
        vpu = vpu_file.name.replace("vpu_", "").replace(".gpkg", "")
        if vpu not in ["prvi"]:
            continue

        print(f"Process {attr} crosswalk for vpu {vpu}")

        # read NextGen divides for the vpu
        shp_vpu = gpd.read_file(vpu_file, layer="divides")
        shp_vpu = shp_vpu[["divide_id", "areasqkm", "geometry"]]

        # read attribute dataset sub-basins given the vpu bounding box
        with fiona.open(attr_dict["file"]) as src:
            crs_attr = src.crs
        bbox = shp_vpu.total_bounds
        bbox_geom = gpd.GeoSeries([box(*bbox)], crs=shp_vpu.crs)
        bbox_reprojected = bbox_geom.to_crs(crs_attr)
        bbox_bounds = bbox_reprojected.total_bounds
        bbox_geom1 = box(*bbox_bounds)
        shp_attr = gpd.read_file(
            attr_dict["file"], layer=attr_dict["layer"], bbox=bbox_geom1
        )

        # rename id column for processing in create_cwt
        shp_attr.rename(columns={attr_dict["id_column"]: "id"}, inplace=True)
        shp_attr = shp_attr[["id", "geometry"]]

        if shp_attr.empty:
            print(f"Warning: no overlapping {attr} subbasins found for vpu {vpu}")
            continue

        # create crosswalk
        cwt1 = create_cwt(
            shp_attr,
            shp_vpu,
            overlap_threshold=threshold,
            overlap_threshold_min=threshold_min,
        )
        if cwt1.empty:
            print(f"No crosswalk created for vpu {vpu}")
            continue

        # reset id column back
        cwt1.rename(columns={"id": attr_dict["id_column"]}, inplace=True)

        # save cwt for the current vpu
        outfile = Path(cwt_dir, "cwt_ngen_" + attr + "_vpu_" + vpu + ".parquet")
        if vpu in ["ak", "hi", "prvi"]:
            outfile = Path(cwt_dir, "cwt_ngen_" + attr + "_" + vpu + ".parquet")
        cwt1.to_parquet(outfile, engine="pyarrow")

        # add to conus dataframe
        if vpu not in ["ak", "hi", "prvi"]:
            cwt1["vpuid"] = vpu
            df_cwt = pd.concat([df_cwt, cwt1])

    # save the combined cwt
    # df_cwt.to_parquet(Path(cwt_dir, 'cwt_ngen_' + attr + '_conus.parquet'), engine='pyarrow')

    return df_cwt

In [28]:
# Create ngen catchment - subbasin crosswalk; process by vpus to reduce memory usage
# attr = 'hydroatlas'
attr = "hlr"
# attr = 'streamcat'
cwt_dir = Path("/home/yuqiong.liu/work/data/ngen_reg", "cwt_ngen_" + attr)
cwt_dir.mkdir(parents=True, exist_ok=True)

df_cwt = process_cwt_by_vpu(attr, cwt_dir, threshold=90.0, threshold_min=20.0)

Process hlr crosswalk for vpu prvi


In [7]:
# analyze crosswalk results
attr_dict = get_attr_info(attr)
cats_total = df_cwt["divide_id"].unique()
df_cwt1 = df_cwt[~df_cwt["nearest_dist_m"].isna()]
cats_unmatched = df_cwt1["divide_id"].unique()
subs_unmatched = df_cwt1[attr_dict["id_column"]].unique()
df_cwt2 = df_cwt[df_cwt["nearest_dist_m"].isna()]
cats_matched = df_cwt2["divide_id"].unique()
counts = df_cwt2["divide_id"].value_counts()
cats_sub1 = counts[counts == 1].index.unique()
cats_sub2 = counts[counts == 2].index.unique()
cats_sub3 = counts[counts == 3].index.unique()
cats_other = counts[counts > 3].index.unique()
print(f"Total number of catchments in CONUS: {len(cats_total)}")
print(
    f"Number of unmatched catchments: {len(cats_unmatched)}, {round(len(cats_unmatched) / len(cats_total) * 100, 2)}%"
)
print(
    f"Number of catchments mapped to 1 {attr.upper()} subbain: {len(cats_sub1)}, {round(len(cats_sub1) / len(cats_total) * 100, 2)}%"
)
print(
    f"Number of catchments mapped to 2 {attr.upper()} subbains: {len(cats_sub2)}, {round(len(cats_sub2) / len(cats_total) * 100, 2)}%"
)
print(
    f"Number of catchments mapped to 3 {attr.upper()} subbains: {len(cats_sub3)}, {round(len(cats_sub3) / len(cats_total) * 100, 2)}%"
)
print(
    f"Number of catchments mapped to more than 3 {attr.upper()} subbains: {len(cats_other)}, {round(len(cats_other) / len(cats_total) * 100, 2)}%"
)

Total number of catchments in CONUS: 831777
Number of unmatched catchments: 4483, 0.54%
Number of catchments mapped to 1 HLR subbain: 793911, 95.45%
Number of catchments mapped to 2 HLR subbains: 31357, 3.77%
Number of catchments mapped to 3 HLR subbains: 1794, 0.22%
Number of catchments mapped to more than 3 HLR subbains: 232, 0.03%


In [ ]:
# plot NextGen catchments and corresponding subbasins (of attribute dataset)
def plot_cats_subs_box(attr: str, cats: list):
    if not cats:
        print("No unmatched catchments found")
        return
    shp_cats = []
    gpkg_ngen = "/home/yuqiong.liu/work/data/gpkg_v2.2/conus_nextgen.gpkg"
    with fiona.open(gpkg_ngen, layer="divides") as src:
        crs_src = src.crs
        for cat in src:
            if cat["properties"]["divide_id"] in cats:
                shp_cats.append(cat)
    gdf_ngen = gpd.GeoDataFrame.from_features(shp_cats, crs=crs_src)

    # get corresponding attribute dataset sub-basins
    with fiona.open(attr_dict["file"], layer=attr_dict["layer"]) as src:
        crs_src = src.crs
    bounds = gdf_ngen.total_bounds
    bbox_geom = box(*bounds)
    bbox_gdf = gpd.GeoDataFrame(geometry=[bbox_geom], crs=gdf_ngen.crs)
    bbox_gdf = bbox_gdf.to_crs(crs_src)
    gdf_subs = gpd.read_file(attr_dict["file"], layer=attr_dict["layer"], bbox=bbox_gdf)

    # plot
    fig, ax = plt.subplots(figsize=(7, 6))
    gdf_subs = gdf_subs.to_crs(epsg=4326)
    gdf_ngen = gdf_ngen.to_crs(epsg=4326)
    if len(gdf_subs) > 0:
        gdf_subs.plot(ax=ax, color="lightgray", edgecolor="black", linewidth=1.0)
    else:
        print(f"No corresponding subbains found for {cats}")
    gdf_ngen.plot(ax=ax, color="none", edgecolor="red", linewidth=0.5)
    # ax.set_axis_off()
    plt.title(f"NextGen catchments (red) and corresponding {attr.upper()} subbasins")
    plt.show()

In [ ]:
# plot NextGen catchments and corresponding subbasins (of attribute dataset)
def plot_cats_subs(attr: str, cats: list, subs: list):
    if not cats:
        print("No unmatched catchments found")
        return
    shp_cats = []
    gpkg_ngen = "/home/yuqiong.liu/work/data/gpkg_v2.2/conus_nextgen.gpkg"
    with fiona.open(gpkg_ngen, layer="divides") as src:
        crs_src = src.crs
        for cat in src:
            if cat["properties"]["divide_id"] in cats:
                shp_cats.append(cat)
    gdf_ngen = gpd.GeoDataFrame.from_features(shp_cats, crs=crs_src)

    # get corresponding attribute dataset sub-basins
    shp_subs = []
    with fiona.open(attr_dict["file"], layer=attr_dict["layer"]) as src:
        crs_src = src.crs
        for cat in src:
            if cat["properties"][attr_dict["id_column"]] in subs:
                shp_subs.append(cat)
    gdf_subs = gpd.GeoDataFrame.from_features(shp_subs, crs=crs_src)

    # plot
    fig, ax = plt.subplots(figsize=(7, 6))
    gdf_subs = gdf_subs.to_crs(epsg=4326)
    gdf_ngen = gdf_ngen.to_crs(epsg=4326)
    gdf_subs.plot(ax=ax, color="lightgray", edgecolor="black", linewidth=1.0)
    gdf_ngen.plot(ax=ax, color="none", edgecolor="red", linewidth=0.5)
    # ax.set_axis_off()
    plt.title(f"NextGen catchments (red) and corresponding {attr.upper()} subbasins")
    plt.show()

In [ ]:
# plot NextGen catchments and corresponding subbasins (of attribute dataset)
def plot_cats_subs_box(attr: str, cats: list):
    if not cats:
        print("No unmatched catchments found")
        return
    shp_cats = []
    gpkg_ngen = "/home/yuqiong.liu/work/data/gpkg_v2.2/conus_nextgen.gpkg"
    with fiona.open(gpkg_ngen, layer="divides") as src:
        crs_src = src.crs
        for cat in src:
            if cat["properties"]["divide_id"] in cats:
                shp_cats.append(cat)
    gdf_ngen = gpd.GeoDataFrame.from_features(shp_cats, crs=crs_src)

    # get corresponding attribute dataset sub-basins
    with fiona.open(attr_dict["file"], layer=attr_dict["layer"]) as src:
        crs_src = src.crs
    bounds = gdf_ngen.total_bounds
    bbox_geom = box(*bounds)
    bbox_gdf = gpd.GeoDataFrame(geometry=[bbox_geom], crs=gdf_ngen.crs)
    bbox_gdf = bbox_gdf.to_crs(crs_src)
    gdf_subs = gpd.read_file(attr_dict["file"], layer=attr_dict["layer"], bbox=bbox_gdf)

    # plot
    fig, ax = plt.subplots(figsize=(7, 6))
    gdf_subs = gdf_subs.to_crs(epsg=4326)
    gdf_ngen = gdf_ngen.to_crs(epsg=4326)
    if len(gdf_subs) > 0:
        gdf_subs.plot(ax=ax, color="lightgray", edgecolor="black", linewidth=1.0)
    else:
        print(f"No corresponding subbains found for {cats}")
    gdf_ngen.plot(ax=ax, color="none", edgecolor="red", linewidth=0.5)
    # ax.set_axis_off()
    plt.title(f"NextGen catchments (red) and corresponding {attr.upper()} subbasins")
    plt.show()

In [ ]:
# plot unmatched NextGen catchments
plot_cats_subs(attr, cats_unmatched.tolist(), subs_unmatched.tolist())

In [ ]:
# plot a random matched catchment
cat1 = random.choice(cats_matched.tolist())
subs = df_cwt[df_cwt["divide_id"] == cat1][attr_dict["id_column"]].to_list()
plot_cats_subs(attr, [cat1], subs)

In [ ]:
# plot the same catchment with bbox method
plot_cats_subs_box(attr, [cat1])

In [ ]:
RUN_THIS_BLOCK = False
if RUN_THIS_BLOCK:
    subs_all = gpd.read_file(attr_dict["file"], layer=attr_dict["layer"])
    subs_all.plot(color="none", edgecolor="red", linewidth=0.5)
    plt.show()